# RSNA Knee Abnormality Detection — Offline Weak-Supervision Baseline

This notebook is designed for the Kaggle Code Competition runtime. It creates an image-based multi-label model from the 58 expert-labeled studies, uses carefully masked report-derived weak labels from the other training reports, and keeps a series-metadata model as a fallback if a DICOM series cannot be decoded.

- No network calls or downloads at runtime.
- The test reports are never read.
- Validation studies are excluded from training, including their report-derived labels.
- Inference failures fall back to a metadata prediction; if that is unavailable, the final values remain finite.
- This is an experimental baseline, not a clinical diagnostic system. Weak report labels are noisy and the validation set is very small.


In [ ]:
import os
import re
import json
import time
import random
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier

try:
    import pydicom
except Exception as exc:
    pydicom = None
    print("WARNING: pydicom is unavailable; image inference will fall back to series metadata.")
    print(type(exc).__name__, str(exc)[:200])

SEED = 42
NUM_SLICES = 8
IMAGE_SIZE = 160
PSEUDO_LABEL_WEIGHT = 0.15
CV_EPOCHS = 4
FINAL_EPOCHS = 4
BATCH_SIZE = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python baseline device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("pydicom available:", pydicom is not None)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
PLANES = ("Sagittal", "Coronal", "Axial")


In [ ]:
def find_competition_root():
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
    ]
    for parent in (Path("/kaggle/input/competitions"), Path("/kaggle/input")):
        if parent.is_dir():
            try:
                candidates.extend([p for p in parent.iterdir() if p.is_dir()])
            except OSError:
                pass

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        required = [
            "train.csv", "train_series.csv", "test.csv",
            "test_series.csv", "sample_submission.csv",
        ]
        if all((candidate / name).is_file() for name in required):
            return candidate
    raise FileNotFoundError(
        "Could not locate the competition input. Attach the official "
        "rsna-knee-abnormality-detection competition data to this notebook."
    )

DATA_ROOT = find_competition_root()
TRAIN_CSV = DATA_ROOT / "train.csv"
TRAIN_SERIES_CSV = DATA_ROOT / "train_series.csv"
TEST_CSV = DATA_ROOT / "test.csv"
TEST_SERIES_CSV = DATA_ROOT / "test_series.csv"
SAMPLE_SUBMISSION_CSV = DATA_ROOT / "sample_submission.csv"

def find_series_root(root: Path, split: str) -> Path:
    candidates = [root / f"{split}_series", root / split]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"No {split} DICOM directory found under {root}")

TRAIN_DICOM_ROOT = find_series_root(DATA_ROOT, "train")
TEST_DICOM_ROOT = find_series_root(DATA_ROOT, "test")

train_df = pd.read_csv(TRAIN_CSV)
train_series_df = pd.read_csv(TRAIN_SERIES_CSV)
test_df = pd.read_csv(TEST_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_CSV)

missing = [target for target in TARGETS if target not in train_df.columns]
if missing:
    raise ValueError(f"train.csv is missing expected targets: {missing}")
for name, frame in (
    ("train.csv", train_df),
    ("train_series.csv", train_series_df),
    ("test.csv", test_df),
    ("test_series.csv", test_series_df),
    ("sample_submission.csv", sample_submission),
):
    if "StudyInstanceUID" not in frame.columns:
        raise ValueError(f"{name} is missing StudyInstanceUID")

if train_df["StudyInstanceUID"].astype(str).duplicated().any():
    raise ValueError("train.csv has duplicate StudyInstanceUID values.")
if test_df["StudyInstanceUID"].astype(str).duplicated().any():
    raise ValueError("test.csv has duplicate StudyInstanceUID values.")

print("Input root:", DATA_ROOT)
print("Training studies:", len(train_df))
print("Fully labeled studies:", int(train_df[TARGETS].notna().all(axis=1).sum()))
print("Test studies:", len(test_df))
print("Training series:", len(train_series_df))
print("Test series:", len(test_series_df))
print("Training DICOM root:", TRAIN_DICOM_ROOT)
print("Test DICOM root:", TEST_DICOM_ROOT)


## Report-derived weak labels

Reports are used only for training. The labeler below marks a target only when a relevant finding is explicitly mentioned; unmentioned findings remain masked rather than being assumed negative. Ligament and meniscus targets additionally require injury/tear language or explicit normal/negative wording. These labels are down-weighted relative to the 58 expert-labeled studies.


In [ ]:
def normalize_report(text):
    text = unicodedata.normalize("NFKD", str(text or ""))
    text = text.encode("ascii", "ignore").decode("ascii").lower()
    # Preserve sentence boundaries so negation cannot bleed into another finding.
    text = re.sub(r"[;:\n\r]+", " . ", text)
    text = re.sub(r"[^a-z0-9.]+", " ", text)
    text = re.sub(r"\.+", " . ", text)
    return re.sub(r"\s+", " ", text).strip()
RULES = {
    "ACL": {
        "anchor": r"\b(?:acl|anterior cruciate ligament|ligamento cruzado anterior|vorderes kreuzband|voorste kruisband|ligament croise anterieur)\b",
        "requires_damage": True,
    },
    "MCL": {
        "anchor": r"\b(?:mcl|medial collateral ligament|medial collateral|ligamento colateral medial|mediales kollateralband|innenband|ligament collat(?:eral)? medial)\b",
        "requires_damage": True,
    },
    "Medial Meniscus": {
        "anchor": r"\b(?:medial meniscus|medial meniscal|internal meniscus|menisco medial|menisco interno|menisque medial|menisque interne|innenmeniskus|medial meniskus)\b",
        "requires_damage": True,
    },
    "Lateral Meniscus": {
        "anchor": r"\b(?:lateral meniscus|lateral meniscal|external meniscus|menisco lateral|menisco externo|menisque lateral|ausserer meniskus|außenmeniskus|lateral meniskus)\b",
        "requires_damage": True,
    },
    "Medial OA": {
        "anchor": r"(?:\b(?:medial|inner|internal|mediale|medialen|compartimento medial|compartimento interno)[a-z0-9 ]{0,65}(?:osteoarthritis|arthrosis|degenerative changes|joint space narrowing|cartilage loss|chondrosis|artrosis|gonarthrosis|osteophytes?)\b|\b(?:osteoarthritis|arthrosis|degenerative changes|joint space narrowing|cartilage loss|chondrosis|artrosis|gonarthrosis|osteophytes?)[a-z0-9 ]{0,65}(?:medial|inner|internal|mediale|medialen|compartimento medial|compartimento interno)\b)",
        "requires_damage": False,
    },
    "Lateral OA": {
        "anchor": r"(?:\b(?:lateral|outer|laterale|lateralen|compartimento lateral|compartimento externo)[a-z0-9 ]{0,65}(?:osteoarthritis|arthrosis|degenerative changes|joint space narrowing|cartilage loss|chondrosis|artrosis|gonarthrosis|osteophytes?)\b|\b(?:osteoarthritis|arthrosis|degenerative changes|joint space narrowing|cartilage loss|chondrosis|artrosis|gonarthrosis|osteophytes?)[a-z0-9 ]{0,65}(?:lateral|outer|laterale|lateralen|compartimento lateral|compartimento externo)\b)",
        "requires_damage": False,
    },
    "PF OA": {
        "anchor": r"(?:\b(?:patellofemoral|femoropatellar|retropatellar|patellar cartilage|trochlear cartilage)[a-z0-9 ]{0,65}(?:osteoarthritis|arthrosis|degenerative changes|cartilage loss|chondrosis|artrosis|osteophytes?|chondral defects?)\b|\b(?:osteoarthritis|arthrosis|degenerative changes|cartilage loss|chondrosis|artrosis|osteophytes?|chondral defects?)[a-z0-9 ]{0,65}(?:patellofemoral|femoropatellar|retropatellar|patellar cartilage|trochlear cartilage)\b)",
        "requires_damage": False,
    },
    "Effusion": {
        "anchor": r"\b(?:effusion|joint fluid|joint effusion|hydrops|derrame articular|derrame|gelenkerguss|erguss|articular fluid)\b",
        "requires_damage": False,
    },
    "Synovitis": {
        "anchor": r"\b(?:synovitis|sinovitis|synovial hypertrophy|synovial thickening|synovial proliferation|synoviale hyperplasie)\b",
        "requires_damage": False,
    },
    "Baker's": {
        "anchor": r"\b(?:baker.?s cyst|popliteal cyst|quiste popliteo|quiste de baker|baker zyste|bakerzyste|poplitealzyste)\b",
        "requires_damage": False,
    },
    "Contusion": {
        "anchor": r"\b(?:bone contusion|bone bruise|marrow edema|bone marrow edema|bone marrow oedema|edema oseo|edema medular|contusion ossea|knochenmarkodem|trabecular injury)\b",
        "requires_damage": False,
    },
    "Fracture": {
        "anchor": r"\b(?:fracture|fractured|fractura|fraktur|fratura|stress fracture|insufficiency fracture|fracture line)\b",
        "requires_damage": False,
    },
}

DAMAGE_WORDS = re.compile(
    r"\b(?:tear|tears|torn|rupture|ruptured|ruptur|ruptura|rotura|"
    r"desgarro|dechirure|riss|ruptuur|scheur|sprain|injury|injured|"
    r"avulsion|discontinuity|partial tear|complete tear|bucket handle|"
    r"radial tear|root tear|mucoid degeneration)\b"
)
NEGATION_PREFIX = re.compile(
    r"(?:\bno\b|\bwithout\b|\bnot\b|\bnegative for\b|"
    r"\babsence of\b|\bno evidence of\b|\bno signs of\b|"
    r"\bsin\b|\bsin signos de\b|\bsin evidencia de\b|"
    r"\bno se observa\b|\bno se identifican\b|\bningun\b|"
    r"\bkein\b|\bkeine\b|\bkeinen\b|\bkeiner\b|\bohne\b|"
    r"\bnicht\b|\bgeen\b|\bgeen aanwijzing voor\b|"
    r"\bpas de\b|\bsans\b|\babsence de\b|\bintact\b|\bnormal\b|\bunremarkable\b|\bpreserved\b|\bwithin normal limits\b)"
)
NEGATION_SUFFIX = re.compile(
    r"\b(?:absent|not seen|not identified|not visualized|not present|"
    r"without evidence|intact|normal|unremarkable|preserved|within normal limits|"
    r"no tear|no rupture|sin rotura|sin desgarro|integro|conservado|"
    r"unauffallig|intakt|kein hinweis)\b"
)

def weak_labels_from_report(report):
    text = normalize_report(report)
    values = np.zeros(len(TARGETS), dtype=np.float32)
    mask = np.zeros(len(TARGETS), dtype=np.float32)

    for target_index, target in enumerate(TARGETS):
        rule = RULES[target]
        for match in re.finditer(rule["anchor"], text):
            # Restrict negation and injury evidence to the current sentence.
            sentence_start = text.rfind(".", 0, match.start()) + 1
            sentence_end = text.find(".", match.end())
            if sentence_end < 0:
                sentence_end = len(text)
            left = text[max(sentence_start, match.start() - 90):match.start()]
            right = text[match.end():min(sentence_end, match.end() + 75)]
            context = left[-80:] + " " + text[match.start():match.end()] + " " + right[:80]

            # A negation must be close to the target phrase, not merely elsewhere
            # in the same report. Up to six intervening tokens covers common
            # constructions such as "no evidence of a tear of the ACL".
            negated_prefix = re.search(
                NEGATION_PREFIX.pattern + r"(?:\s+[a-z0-9]+){0,6}\s*$",
                left,
            ) is not None
            negated = (
                negated_prefix
                or NEGATION_PREFIX.search(match.group(0)) is not None
                or NEGATION_SUFFIX.search(right) is not None
            )
            if rule["requires_damage"]:
                has_damage = DAMAGE_WORDS.search(context) is not None
                if negated:
                    values[target_index] = 0.0
                    mask[target_index] = 1.0
                    break
                if has_damage:
                    values[target_index] = 1.0
                    mask[target_index] = 1.0
                    break
                # Merely mentioning a ligament or meniscus is not a positive.
                continue

            values[target_index] = 0.0 if negated else 1.0
            mask[target_index] = 1.0
            break

    return values, mask

# Small sanity check on representative positive, negative and unknown text.
examples = {
    "The ACL is torn and there is a joint effusion.": {"ACL": 1, "Effusion": 1},
    "The ACL is intact. No joint effusion.": {"ACL": 0, "Effusion": 0},
    "Menisci are mentioned without tear details.": {},
}
for report, expected in examples.items():
    vals, known = weak_labels_from_report(report)
    observed = {
        target: int(vals[TARGETS.index(target)])
        for target in expected
        if known[TARGETS.index(target)] > 0
    }
    assert observed == expected, (report, observed, expected)
print("Report-label sanity checks passed.")


In [ ]:
def build_training_targets(frame):
    n = len(frame)
    y = np.zeros((n, len(TARGETS)), dtype=np.float32)
    mask = np.zeros_like(y)
    weights = np.zeros_like(y)
    gold = frame[TARGETS].notna().all(axis=1).to_numpy()

    for i, row in enumerate(frame.itertuples(index=False, name=None)):
        # Use column lookup from the dataframe for robust target ordering.
        if gold[i]:
            y[i] = frame.iloc[i][TARGETS].to_numpy(dtype=np.float32)
            mask[i] = 1.0
            weights[i] = 1.0
        else:
            weak_values, weak_mask = weak_labels_from_report(frame.iloc[i].get("Report", ""))
            y[i] = np.where(weak_values >= 0.5, 0.90, 0.10)
            mask[i] = weak_mask
            weights[i] = weak_mask * PSEUDO_LABEL_WEIGHT

    # Report-derived targets are weak supervision, not replacements for expert labels.
    weights[gold] = 1.0
    return y, mask, weights, gold

y_train, label_mask, label_weights, gold_mask = build_training_targets(train_df)
known_counts = label_mask.sum(axis=0).astype(int)
weak_study_count = int((label_mask[~gold_mask].sum(axis=1) > 0).sum())
print("Expert-labeled studies:", int(gold_mask.sum()))
print("Studies with at least one report-derived weak target:", weak_study_count)
print("Known targets by class:")
print(pd.Series(known_counts, index=TARGETS).to_string())
print("Weak-label positive rate among known pseudo-targets:")
for j, target in enumerate(TARGETS):
    select = (~gold_mask) & (label_mask[:, j] > 0)
    rate = float((y_train[select, j] >= 0.5).mean()) if select.any() else float("nan")
    print(f"  {target}: n={int(select.sum())}, positive_rate={rate:.3f}")


## MRI slice extraction and caching

For each study, this pipeline selects one fluid-sensitive series, preferring sagittal then coronal then axial. It reads DICOM headers to order slices physically and decodes only eight evenly spaced slices. Failed series are logged and excluded from image-model training; the output pipeline still produces a valid submission via the metadata fallback.


In [ ]:
def series_candidates(study_uid, series_frame):
    rows = series_frame[
        series_frame["StudyInstanceUID"].astype(str) == str(study_uid)
    ].copy()
    if rows.empty:
        return []

    if "Fluid_Sensitive" in rows.columns:
        fluid = pd.to_numeric(rows["Fluid_Sensitive"], errors="coerce").fillna(0)
        rows["_fluid_rank"] = np.where(fluid.to_numpy() == 1, 0, 10)
    else:
        rows["_fluid_rank"] = 10

    plane_rank = {plane: index for index, plane in enumerate(PLANES)}
    rows["_plane_rank"] = rows.get(
        "Anatomical_Plane", pd.Series("", index=rows.index)
    ).map(plane_rank).fillna(9)

    if "SeriesInstanceUID" not in rows.columns:
        return []
    rows = rows.sort_values(
        ["_fluid_rank", "_plane_rank", "SeriesInstanceUID"],
        kind="stable",
    )
    return [str(value) for value in rows["SeriesInstanceUID"].dropna().tolist()]


def list_dicom_files(series_dir):
    series_dir = Path(series_dir)
    if not series_dir.is_dir():
        raise FileNotFoundError(f"Series directory not found: {series_dir}")
    files = sorted(p for p in series_dir.iterdir() if p.is_file() and not p.name.startswith("."))
    if not files:
        raise FileNotFoundError(f"No files in series directory: {series_dir}")
    return files


def read_sampled_series(series_dir):
    if pydicom is None:
        raise RuntimeError("pydicom is not available")
    files = list_dicom_files(series_dir)
    records = []

    for fallback_index, path in enumerate(files):
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
            position = getattr(ds, "ImagePositionPatient", None)
            orientation = getattr(ds, "ImageOrientationPatient", None)
            if position is not None and orientation is not None and len(orientation) >= 6:
                position = np.asarray(position, dtype=np.float64)
                orientation = np.asarray(orientation, dtype=np.float64)
                normal = np.cross(orientation[:3], orientation[3:6])
                sort_position = float(np.dot(position, normal))
            else:
                sort_position = float(getattr(ds, "InstanceNumber", fallback_index))
            records.append((sort_position, fallback_index, path))
        except Exception:
            # One malformed header should not discard otherwise readable slices.
            continue

    if not records:
        raise RuntimeError(f"Could not read DICOM headers in {series_dir}")
    records.sort(key=lambda item: (item[0], item[1]))
    indices = np.rint(np.linspace(0, len(records) - 1, NUM_SLICES)).astype(int)

    output = np.zeros((NUM_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    for slot, record_index in enumerate(indices):
        path = records[int(record_index)][2]
        ds = pydicom.dcmread(path, force=True)
        pixels = np.asarray(ds.pixel_array)
        if pixels.ndim == 3:
            pixels = pixels[0]
        if pixels.ndim != 2:
            raise ValueError(f"Unexpected pixel dimensions {pixels.shape} in {path.name}")

        pixels = pixels.astype(np.float32)
        if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")).upper() == "MONOCHROME1":
            pixels = pixels.max() + pixels.min() - pixels

        if not np.isfinite(pixels).all():
            raise ValueError(f"Non-finite pixel data in {path.name}")
        low, high = np.percentile(pixels, [1, 99])
        if high <= low:
            normalized = np.zeros_like(pixels, dtype=np.float32)
        else:
            normalized = np.clip((pixels - low) / (high - low), 0.0, 1.0)

        scaled = (normalized * 255.0).round().astype(np.uint8)
        resized = Image.fromarray(scaled).resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR
        )
        output[slot] = np.asarray(resized, dtype=np.uint8)

    return output


def build_image_cache(studies_frame, series_frame, dicom_root, cache_path, label):
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache = np.lib.format.open_memmap(
        cache_path,
        mode="w+",
        dtype=np.uint8,
        shape=(len(studies_frame), NUM_SLICES, IMAGE_SIZE, IMAGE_SIZE),
    )
    cache[:] = 0
    success = np.zeros(len(studies_frame), dtype=bool)
    errors = []
    started = time.time()

    for i, study_uid in enumerate(studies_frame["StudyInstanceUID"].astype(str)):
        last_error = None
        candidates = series_candidates(study_uid, series_frame)
        for series_uid in candidates:
            series_dir = Path(dicom_root) / study_uid / series_uid
            try:
                sampled = read_sampled_series(series_dir)
                cache[i] = sampled
                success[i] = True
                break
            except Exception as exc:
                last_error = f"{type(exc).__name__}: {str(exc)[:160]}"

        if not success[i]:
            cache[i] = 0
            errors.append((study_uid, last_error or "No matching series"))
        if (i + 1) % 100 == 0 or i + 1 == len(studies_frame):
            elapsed = time.time() - started
            print(f"{label}: {i + 1}/{len(studies_frame)} studies; "
                  f"decoded={int(success[:i+1].sum())}; elapsed={elapsed/60:.1f} min")

    cache.flush()
    del cache
    cache = np.load(cache_path, mmap_mode="r")
    print(f"{label}: success {int(success.sum())}/{len(success)}, failures {len(errors)}")
    if errors:
        print("First decoding errors:")
        for item in errors[:5]:
            print(" ", item)
    return cache, success

CACHE_ROOT = Path("/kaggle/working/rsna_knee_cache")
train_images, train_image_ok = build_image_cache(
    train_df, train_series_df, TRAIN_DICOM_ROOT,
    CACHE_ROOT / "train_images.npy", "TRAIN",
)
test_images, test_image_ok = build_image_cache(
    test_df, test_series_df, TEST_DICOM_ROOT,
    CACHE_ROOT / "test_images.npy", "TEST",
)


## Metadata fallback

The fallback uses only the provided per-study series metadata and report-derived training targets. It is not a replacement for image analysis, but ensures that a failed or unsupported DICOM decoder does not prevent the notebook from writing a valid submission.


In [ ]:
def build_series_features(studies_frame, series_frame):
    records = []
    series_frame = series_frame.copy()
    series_frame["StudyInstanceUID"] = series_frame["StudyInstanceUID"].astype(str)

    if "Anatomical_Plane" not in series_frame.columns:
        series_frame["Anatomical_Plane"] = ""
    if "Fluid_Sensitive" not in series_frame.columns:
        series_frame["Fluid_Sensitive"] = 0
    if "Fat_Suppression" not in series_frame.columns:
        series_frame["Fat_Suppression"] = 0

    series_frame["_plane"] = series_frame["Anatomical_Plane"].astype(str).str.lower()
    series_frame["_fluid"] = pd.to_numeric(
        series_frame["Fluid_Sensitive"], errors="coerce"
    ).fillna(0).astype(int)
    series_frame["_fat"] = pd.to_numeric(
        series_frame["Fat_Suppression"], errors="coerce"
    ).fillna(0).astype(int)

    for study_uid in studies_frame["StudyInstanceUID"].astype(str):
        group = series_frame[series_frame["StudyInstanceUID"] == study_uid]
        row = {"n_series": float(len(group))}
        row["n_fluid_sensitive"] = float(group["_fluid"].sum())
        row["n_fat_suppressed"] = float(group["_fat"].sum())

        for plane in PLANES:
            subset = group[group["_plane"] == plane.lower()]
            prefix = plane.lower()
            row[f"{prefix}_series"] = float(len(subset))
            row[f"{prefix}_fluid"] = float(subset["_fluid"].sum())
            row[f"{prefix}_fat"] = float(subset["_fat"].sum())
            row[f"{prefix}_fluid_fat"] = float(
                ((subset["_fluid"] == 1) & (subset["_fat"] == 1)).sum()
            )
        records.append(row)

    features = pd.DataFrame(records).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return features.astype(np.float32)

X_meta_train = build_series_features(train_df, train_series_df)
X_meta_test = build_series_features(test_df, test_series_df)
if list(X_meta_train.columns) != list(X_meta_test.columns):
    raise ValueError("Training and test metadata feature columns do not match.")

def fit_metadata_fallback(X_train, y, mask, weights, X_test):
    predictions = np.full((len(X_test), len(TARGETS)), 0.5, dtype=np.float32)
    models = []
    for j, target in enumerate(TARGETS):
        usable = (mask[:, j] > 0) & (weights[:, j] > 0)
        if not usable.any():
            models.append(None)
            continue

        target_binary = (y[usable, j] >= 0.5).astype(np.int32)
        sample_weight = weights[usable, j].astype(np.float64)
        weighted_prior = float(
            (target_binary * sample_weight).sum() / max(sample_weight.sum(), 1e-8)
        )
        if np.unique(target_binary).size < 2:
            predictions[:, j] = np.clip(weighted_prior, 0.01, 0.99)
            models.append(None)
            continue

        model = ExtraTreesClassifier(
            n_estimators=160,
            min_samples_leaf=6,
            max_features=0.8,
            n_jobs=-1,
            random_state=SEED + j,
        )
        model.fit(
            X_train.loc[usable].to_numpy(dtype=np.float32),
            target_binary,
            sample_weight=sample_weight,
        )
        positive_col = list(model.classes_).index(1)
        predictions[:, j] = model.predict_proba(
            X_test.to_numpy(dtype=np.float32)
        )[:, positive_col]
        models.append(model)

    return np.clip(predictions, 0.01, 0.99), models

metadata_predictions, metadata_models = fit_metadata_fallback(
    X_meta_train, y_train, label_mask, label_weights, X_meta_test
)
print("Metadata fallback is ready.")


## Validate and train the image model

We use one study-level validation fold from the 58 expert-labeled studies as a diagnostic. A validation study is removed from the entire training index so its report cannot leak into the weak-label training set. Because there are only 58 expert labels, this fold is a noisy diagnostic—not a reliable estimate of leaderboard performance.


In [ ]:
class CachedStudyDataset(Dataset):
    def __init__(self, image_cache, labels, mask, weights, indices):
        self.image_cache = image_cache
        self.labels = labels
        self.mask = mask
        self.weights = weights
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        i = int(self.indices[item])
        image = np.asarray(self.image_cache[i], dtype=np.float32).copy() / 255.0
        return (
            torch.from_numpy(image),
            torch.from_numpy(self.labels[i].astype(np.float32)),
            torch.from_numpy(self.mask[i].astype(np.float32)),
            torch.from_numpy(self.weights[i].astype(np.float32)),
        )


class SliceCNN(nn.Module):
    def __init__(self, num_targets=12):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.SiLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.SiLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.head = nn.Sequential(
            nn.Linear(64, 96),
            nn.SiLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(96, num_targets),
        )

    def forward(self, images):
        if images.ndim != 4:
            raise ValueError("Expected [batch, slices, height, width].")
        batch_size, slices, height, width = images.shape
        per_slice = images.reshape(batch_size * slices, 1, height, width)
        features = self.encoder(per_slice).reshape(batch_size, slices, -1)
        return self.head(features.mean(dim=1))


def train_model(images, y, mask, weights, indices, epochs, label):
    dataset = CachedStudyDataset(images, y, mask, weights, indices)
    if len(dataset) == 0:
        raise ValueError("No decodable training studies with usable labels.")
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )
    model = SliceCNN(num_targets=len(TARGETS)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
    losses = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_weight = 0.0
        start = time.time()

        for image, labels, target_mask, target_weights in loader:
            image = image.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            effective_weight = (
                target_mask.to(DEVICE, non_blocking=True)
                * target_weights.to(DEVICE, non_blocking=True)
            )
            optimizer.zero_grad(set_to_none=True)
            logits = model(image)
            per_target_loss = nn.functional.binary_cross_entropy_with_logits(
                logits, labels, reduction="none"
            )
            denominator = effective_weight.sum()
            if float(denominator.detach().item()) <= 0:
                continue
            loss = (per_target_loss * effective_weight).sum() / denominator
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()

            total_loss += float((per_target_loss.detach() * effective_weight).sum().item())
            total_weight += float(denominator.detach().item())

        epoch_loss = total_loss / max(total_weight, 1e-8)
        losses.append(epoch_loss)
        print(
            f"{label} epoch {epoch}/{epochs}: "
            f"weighted_loss={epoch_loss:.5f}, {time.time()-start:.1f}s"
        )

    return model, losses


def predict_image_model(model, image_cache, image_ok, batch_size=16):
    predictions = np.full((len(image_ok), len(TARGETS)), 0.5, dtype=np.float32)
    valid_indices = np.flatnonzero(image_ok)
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(valid_indices), batch_size):
            ids = valid_indices[start:start + batch_size]
            batch = np.asarray(image_cache[ids], dtype=np.float32).copy() / 255.0
            logits = model(torch.from_numpy(batch).to(DEVICE))
            predictions[ids] = torch.sigmoid(logits).cpu().numpy()
    return predictions

gold_indices = np.flatnonzero(gold_mask)
if len(gold_indices) < 5:
    raise ValueError("Too few expert-labeled studies for a study-level validation split.")

splitter = KFold(n_splits=min(5, len(gold_indices)), shuffle=True, random_state=SEED)
_, validation_positions = next(splitter.split(gold_indices))
validation_indices = gold_indices[validation_positions]
validation_uid_set = set(train_df.iloc[validation_indices]["StudyInstanceUID"].astype(str))
all_indices = np.arange(len(train_df))
is_validation_study = train_df["StudyInstanceUID"].astype(str).isin(validation_uid_set).to_numpy()

image_weights = label_weights.copy()
image_weights[~train_image_ok] = 0.0

cv_train_indices = all_indices[
    (~is_validation_study)
    & train_image_ok
    & (image_weights.sum(axis=1) > 0)
]
cv_valid_indices = validation_indices[train_image_ok[validation_indices]]

print("CV training studies:", len(cv_train_indices))
print("CV expert validation studies:", len(validation_indices))
print("CV validation studies with readable images:", len(cv_valid_indices))


In [ ]:
cv_model = None
cv_metrics = {}
if int(train_image_ok.sum()) >= 30 and len(cv_train_indices) >= 20:
    cv_model, cv_history = train_model(
        train_images, y_train, label_mask, image_weights,
        cv_train_indices, CV_EPOCHS, "CV",
    )
    cv_predictions = predict_image_model(cv_model, train_images, train_image_ok)

    actual = train_df.iloc[validation_indices][TARGETS].to_numpy(dtype=np.float32)
    predicted = cv_predictions[validation_indices]
    per_target_auc = {}
    available_auc = []
    for j, target in enumerate(TARGETS):
        if np.unique(actual[:, j]).size == 2:
            score = float(roc_auc_score(actual[:, j], predicted[:, j]))
            per_target_auc[target] = score
            available_auc.append(score)
        else:
            per_target_auc[target] = None
    cv_metrics = {
        "validation_studies": len(validation_indices),
        "readable_validation_images": int(train_image_ok[validation_indices].sum()),
        "evaluated_targets": len(available_auc),
        "macro_auc_on_evaluable_targets": (
            float(np.mean(available_auc)) if available_auc else None
        ),
        "per_target_auc": per_target_auc,
        "validation_loss_history": cv_history,
    }
    print("Small-fold diagnostic:")
    print(json.dumps(cv_metrics, indent=2))
else:
    print("Not enough decoded training images for image-model CV; metadata fallback will be used.")


## Final fit, prediction and submission validation

The final model uses all successfully decoded training studies and all 58 expert labels, with report-derived targets down-weighted. Any unparseable DICOM study receives the metadata fallback. The test Report field is never accessed.


In [ ]:
final_model = None
final_history = []
if int(train_image_ok.sum()) >= 30:
    final_indices = np.flatnonzero(
        train_image_ok & (image_weights.sum(axis=1) > 0)
    )
    try:
        final_model, final_history = train_model(
            train_images, y_train, label_mask, image_weights,
            final_indices, FINAL_EPOCHS, "FINAL",
        )
    except Exception as exc:
        print("Image model training failed; using metadata fallback:", type(exc).__name__, str(exc)[:300])
        final_model = None
else:
    print("Too few readable training studies; using metadata fallback.")

final_predictions = metadata_predictions.copy()
if final_model is not None:
    image_predictions = predict_image_model(final_model, test_images, test_image_ok)
    readable_test = np.flatnonzero(test_image_ok)
    final_predictions[readable_test] = image_predictions[readable_test]
    print(
        f"Image model used for {len(readable_test)}/{len(test_df)} test studies; "
        f"metadata fallback used for {int((~test_image_ok).sum())}."
    )
else:
    print("Metadata fallback used for all test studies.")

final_predictions = np.nan_to_num(
    final_predictions.astype(np.float64),
    nan=0.5, posinf=0.5, neginf=0.5,
)
final_predictions = np.clip(final_predictions, 0.0, 1.0)

prediction_frame = pd.DataFrame(
    final_predictions,
    columns=TARGETS,
)
prediction_frame.insert(
    0, "StudyInstanceUID", test_df["StudyInstanceUID"].astype(str).to_numpy()
)

if prediction_frame["StudyInstanceUID"].duplicated().any():
    raise ValueError("Predictions have duplicate StudyInstanceUID values.")
if sample_submission["StudyInstanceUID"].astype(str).duplicated().any():
    raise ValueError("Submission template has duplicate StudyInstanceUID values.")

lookup = prediction_frame.set_index("StudyInstanceUID")
template_uids = sample_submission["StudyInstanceUID"].astype(str)
missing_uids = sorted(set(template_uids) - set(lookup.index))
extra_uids = sorted(set(lookup.index) - set(template_uids))
if missing_uids or extra_uids:
    raise ValueError(
        f"Test/template UID mismatch: {len(missing_uids)} missing, {len(extra_uids)} extra."
    )

submission = sample_submission.copy()
submission.loc[:, TARGETS] = lookup.loc[template_uids, TARGETS].to_numpy(dtype=np.float64)
values = submission[TARGETS].to_numpy(dtype=np.float64)

if submission.columns.tolist() != ["StudyInstanceUID"] + TARGETS:
    raise ValueError("Final submission columns are not in the required order.")
if not np.isfinite(values).all():
    raise ValueError("Final submission contains NaN or infinite values.")
if np.any((values < 0.0) | (values > 1.0)):
    raise ValueError("Final submission probabilities fall outside [0, 1].")
if len(submission) != len(sample_submission):
    raise ValueError("Final submission row count does not match the template.")

OUTPUT_PATH = Path("/kaggle/working/submission.csv")
submission.to_csv(OUTPUT_PATH, index=False)
print("Wrote:", OUTPUT_PATH)
print("Shape:", submission.shape)
print("First rows:")
print(submission.head(3).to_string(index=False))
print("File bytes:", OUTPUT_PATH.stat().st_size)


## Reproducibility note

The console log records which DICOM studies decoded successfully, the report-target coverage, a small study-level validation diagnostic, and the final submission shape. Results may vary across Kaggle hardware/library images; inspect decoder failures before interpreting scores.
